In [1]:
import numpy as np
import pandas as pd

df = pd.read_csv('cars.csv')
df = df.drop(df.columns[0], axis=1)

In [2]:
df.shape

(8128, 4)

In [3]:
df.sample(5)

,km_driven,fuel,owner,selling_price
3518,45000,Diesel,First Owner,3200000
7980,30000,Petrol,First Owner,300000
7895,82000,Petrol,Second Owner,340000
3636,50000,Diesel,Second Owner,95000
5314,10000,Diesel,First Owner,1200000


In [4]:
df.isnull().sum()

km_driven        0
fuel             0
owner            0
selling_price    0
dtype: int64

In [6]:
owner_mapping = {
    'First Owner': 0,
    'Second Owner': 1,
    'Third Owner': 2,
    'Fourth & Above Owner': 3,
    'Test Drive Car': 4,
}

df['owner'] = df['owner'].map(owner_mapping)

# One-hot encode 'fuel'
df = pd.get_dummies(df, columns=['fuel'], drop_first=True, dtype=int)

In [7]:
feature_cols = df.drop('selling_price', axis=1).columns
X = df[feature_cols].values
y = df['selling_price'].values.reshape(-1, 1)


In [9]:
# Normalization

price_mean = np.mean(y)
price_std = np.std(y)
y_scaled = (y - price_mean) / price_std

# 3. Standardize features
X_mean = np.mean(X, axis=0)
X_std = np.std(X, axis=0)
X_std[X_std == 0] = 1
X_scaled = (X - X_mean) / X_std

In [10]:
X_scaled = np.c_[np.ones(X_scaled.shape[0]), X_scaled]

# 4. Train with Gradient Descent
W = np.zeros((X_scaled.shape[1], 1))
learning_rate = 0.01
epochs = 2000

In [11]:
for i in range(epochs):
  y_pred_scaled = X_scaled @ W
  error = y_pred_scaled - y_scaled
  gradient = (1 / len(y_scaled)) * (X_scaled.T @ error)
  W = W - learning_rate * gradient


In [12]:
# 5. Convert dataset predictions back to real selling prices
predicted_prices = (X_scaled @ W) * price_std + price_mean

results = pd.DataFrame({
    'Actual_Price': y.flatten(),
    'Predicted_Price': predicted_prices.flatten().round(0),
})

In [13]:
def predict_car_price(km_driven, owner, fuel):
  new_car = pd.DataFrame([{'km_driven': km_driven, 'owner': owner}])
  for col in feature_cols:
    if col.startswith('fuel_'):
      new_car[col] = 1 if col == f'fuel_{fuel}' else 0

  new_X = new_car[feature_cols].values
  new_X_scaled = (new_X - X_mean) / X_std
  new_X_scaled = np.c_[np.ones(new_X_scaled.shape[0]), new_X_scaled]

  pred_scaled = new_X_scaled @ W
  return float((pred_scaled * price_std + price_mean)[0, 0])


In [14]:
sample_price = predict_car_price(km_driven=50000, owner=0, fuel='Diesel')
print(f'\nEstimated Selling Price: ₹{sample_price:,.0f}')


Estimated Selling Price: ₹992,336
